In [5]:
# IMPORTS
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

import pandas as pd
import requests
from io import StringIO
from datetime import datetime
import os

In [6]:
# FUNÇÃO INPUT
def input_bronze(dataset: str) -> pd.DataFrame:
    url = f"https://dados.anvisa.gov.br/dados/VigiMed_{dataset}.csv"
    resp = requests.get(url, verify=False)
    data = StringIO(resp.text)
    df = pd.read_csv(
        data,
        sep=';',
        encoding='latin-1',
        dtype=str,
        quoting=3,
        on_bad_lines='error',
        low_memory=False
    )
    return df

In [14]:
# FUNÇÃO OUTPUT
def output_bronze(df: pd.DataFrame, dataset: str) -> str:
    layer = "01_bronze"
    folder = os.path.join("..", "data", layer, dataset)  # subpasta por dataset
    os.makedirs(folder, exist_ok=True)

    # remove todos os arquivos antigos da pasta
    for f in os.listdir(folder):
        file_path = os.path.join(folder, f)
        if os.path.isfile(file_path):
            print(f"Remover arquivo: {file_path}")
            os.remove(file_path)

    # nome do arquivo com data
    date = datetime.now().strftime("%d%m%Y")
    output_path = os.path.join(folder, f"{dataset}_{date}.parquet")

    df.to_parquet(output_path, index=False, engine="pyarrow")
    return output_path  

In [ ]:
# EXECUÇÃO
for dataset in ["Medicamentos", "Notificacoes", "Reacoes"]:
    df = input_bronze(dataset)
    path = output_bronze(df, dataset)
    print(f"Salvar Bronze: {path}")

Remover arquivo: ..\data\01_bronze\Medicamentos\Medicamentos_29092025.parquet
Salvo Bronze: ..\data\01_bronze\Medicamentos\Medicamentos_29092025.parquet
Remover arquivo: ..\data\01_bronze\Notificacoes\Notificacoes_29092025.parquet
Salvo Bronze: ..\data\01_bronze\Notificacoes\Notificacoes_29092025.parquet
Remover arquivo: ..\data\01_bronze\Reacoes\Reacoes_29092025.parquet
Salvo Bronze: ..\data\01_bronze\Reacoes\Reacoes_29092025.parquet
